# 🤖 LLM Fundamentals for Business Managers
### Essential Knowledge (15-20 minutes)

---
## ⚙️ Setup - Run This First

In [1]:
!pip install litellm tiktoken python-dotenv ipython litellm -q

import os
from dotenv import load_dotenv
from litellm import completion
import tiktoken
from IPython.display import Markdown, display, HTML
import litellm

load_dotenv()

# Styling functions from enhanced version
NAVY = "#0B1F3A"
ICE = "#EAF2FF"
GOLD = "#B08D57"

def banner(text, color=NAVY):
    display(HTML(f"""
    <div style="background:{color};color:{ICE};padding:14px 22px;border-radius:8px;
                font-family:'Segoe UI',sans-serif;font-size:19px;font-weight:600;
                margin:14px 0 6px 0;letter-spacing:0.2px;">
        {text}
    </div>
    """))

def insight(text):
    display(HTML(f"""
    <div style="background:{ICE};border-left:5px solid {GOLD};padding:14px 20px;
                border-radius:4px;font-family:'Segoe UI',sans-serif;font-size:15.5px;
                color:{NAVY};margin:10px 0 22px 0;">
        <b>💡 Business Insight:</b> {text}
    </div>
    """))

def warning_box(text):
    display(HTML(f"""
    <div style="background:#FBEAEA;border-left:5px solid #B23B3B;padding:14px 20px;
                border-radius:4px;font-family:'Segoe UI',sans-serif;font-size:15.5px;
                color:#5A1A1A;margin:10px 0 22px 0;">
        <b>⚠️ Critical Warning:</b> {text}
    </div>
    """))

print("✅ Setup complete. You are ready to go!")


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Users\vivik\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


✅ Setup complete. You are ready to go!


---
## 1. What Are LLMs Really Doing?

Large Language Models are **advanced next-word predictors**. 

They don’t truly “understand” — they predict the most likely next token based on patterns learned during training.

---
## 2. Tokens — The Real Unit of Cost

In [2]:
enc = tiktoken.encoding_for_model("gpt-4o")

examples = {
    "One-line status update": "Backup job 4471 completed successfully at 2:03 AM.",
    "Short incident note": "Nightly backup for East Region cluster failed due to network timeout.",
    "Full postmortem excerpt": "At 01:47 AM, the scheduled backup for the East Region storage cluster failed..."
}

banner("Token Cost by Document Type")

for label, text in examples.items():
    words = len(text.split())
    tokens = len(enc.encode(text))
    print(f"{label:<35} {words:>3} words → {tokens:>3} tokens")

insight("You are billed **per token**, not per word. A 10-page document (~5,000 tokens) summarized with gpt-4o-mini costs less than a penny.")

One-line status update                8 words →  14 tokens
Short incident note                  11 words →  13 tokens
Full postmortem excerpt              13 words →  18 tokens


---
## 3. Next Word Prediction

In [3]:
def show_probability(prompt):
    resp = completion(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=1,
        logprobs=True,
        top_logprobs=5
    )
    print(f"Prompt: '{prompt}'")
    for item in resp.choices[0].logprobs.content[0].top_logprobs:
        prob = round((2.71828 ** item.logprob) * 100, 1)
        print(f"  {item.token:<15} {prob:6.1f}%")

show_probability("The capital of France is")
show_probability("A good way to reduce IT support costs is")
show_probability("A man was running so fast because")

Prompt: 'The capital of France is'
  Paris            100.0%
  The                0.0%
   Paris             0.0%
  the                0.0%
  Par                0.0%
Prompt: 'A good way to reduce IT support costs is'
  A                 47.5%
  to                32.6%
  implement          8.8%
  There              4.8%
  Reduc              2.9%
Prompt: 'A man was running so fast because'
  there             81.7%
  he                17.7%
  There              0.3%
  A                  0.1%
  The                0.1%


---
## 4. Temperature: Predictable vs Creative

In [4]:
prompt = "Draft a one-paragraph internal summary for leadership about a recurring backup failure on the East Region cluster."

for temp in [0.0, 0.7, 1.4]:
    banner(f"Temperature = {temp} {'(Strict & Consistent)' if temp == 0.0 else '(Balanced)' if temp == 0.7 else '(Creative & Exploratory)'}", color=GOLD if temp > 0.5 else NAVY)
    resp = completion(model="gpt-4o", messages=[{"role": "user", "content": prompt}], temperature=temp)
    display(Markdown(resp.choices[0].message.content.strip()))
    print("─" * 80)

insight("**Low temperature (0.0-0.3)** → Best for finance, legal, compliance, reports.<br>**Higher temperature (0.7-1.0)** → Best for brainstorming, marketing, creative content.")

Subject: Summary of Recurring Backup Failure on East Region Cluster

Dear Leadership Team,

We are currently experiencing a recurring backup failure on the East Region cluster, which has been impacting our data integrity and recovery processes. The issue was first identified on [insert date], and subsequent investigations have revealed that the failures are primarily due to [insert identified cause, e.g., network latency, hardware malfunctions, or software bugs]. Our IT team is actively working with [insert relevant vendors or departments] to diagnose the root cause and implement a robust solution. In the interim, we have initiated temporary measures to mitigate data loss risks, including [insert temporary measures, e.g., increasing backup frequency, utilizing alternative storage solutions]. We are committed to resolving this issue promptly and will provide regular updates on our progress. Your understanding and support during this critical time are greatly appreciated.

Best regards,

[Your Name]
[Your Position]

────────────────────────────────────────────────────────────────────────────────


Subject: Summary of Recurring Backup Failure on East Region Cluster

Dear Leadership Team,

We have identified a recurring issue with the backup processes on the East Region cluster, which has been failing consistently over the past two weeks. The failures are primarily occurring during the data transfer phase, where an unexpected bottleneck within the network infrastructure has been detected. Preliminary investigations suggest that the bandwidth allocation for backup operations is being inadvertently throttled due to recent network configuration changes aimed at optimizing overall system performance. As a result, this has led to incomplete backups and potential data integrity risks. Our IT team is actively working on resolving the issue by recalibrating the network settings to ensure sufficient resources are allocated for backup operations. We are also exploring alternative solutions, such as scheduling backups during off-peak hours to minimize network congestion. We recognize the critical importance of reliable backup systems and are committed to restoring full functionality promptly. Further updates will be provided as we progress with our remediation efforts.

Best regards,

[Your Name]  
[Your Position]

────────────────────────────────────────────────────────────────────────────────


Subject: Summary of Recurring Backup Failures in East Region Cluster

Since the east quarter of last month, we’ve been observing continuous backup failures within our East Region cluster, compromising the integrity and reliability of our data preservation processes. These failures have instigated potential security risks, extended core operation downtimes, and negatively impacted our disaster recovery capabilities. Preliminary investigations attribute the root cause to potentially outdated firmware that seems incompatible with current system requirements, as well as insufficient allocation of resources meant to facilitate seamless data transfer among storage nodes, thus overwhelming the network partition managing backup queries. We're prioritizing immediate firmware updates and network reconfiguration consultation, doubling server monitoring routines as preventative measures, and delineating two existing Press services not priority-aligned. We encourage cross-department collaborations urgently aimed at resolving forthcoming challenges safeguarding our service quality vict.

────────────────────────────────────────────────────────────────────────────────


---
## 5. ⚠️ The Hallucination Warning

In [5]:
warning_box("LLMs can generate fluent, confident, but completely false information.<br><br>"
            "<b>Real business risks:</b> Made-up policies, fake court cases, incorrect numbers, non-existent features.<br><br>"
            "<b>Rule:</b> Always verify critical outputs with human oversight.")

---
## 6. System Prompts — Turn One Assistant into Many Experts

In [6]:
def ask(system_prompt, user_prompt, model="gpt-4o-mini", temperature=0.0):
    resp = completion(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=temperature
    )
    return resp.choices[0].message.content.strip()

ticket = "Customer's nightly backup failed three times due to storage quota limit. Resolved by increasing capacity."

banner("Internal Technical Summarizer")
internal = "You are a concise IT support analyst. Be technical, factual, and structured."
display(Markdown(ask(internal, f"Summarize: {ticket}")))

banner("Customer-Facing Communicator", color=GOLD)
customer = "You are a warm, reassuring customer success manager. Avoid blame and use simple language."
display(Markdown(ask(customer, f"Write a short email to the customer about: {ticket}")))

insight("A good **system prompt** is like giving a new hire a clear job description. It dramatically changes tone, depth, and focus.")

**Issue Summary:**
- **Problem:** Nightly backup failed three times.
- **Cause:** Storage quota limit exceeded.
- **Resolution:** Increased storage capacity to resolve the issue.

Subject: Update on Your Nightly Backup

Hi [Customer's Name],

I hope this message finds you well! I wanted to let you know that we noticed your nightly backup had failed three times due to reaching the storage quota limit. 

Good news! We’ve increased your storage capacity, so your backups should run smoothly from now on. 

If you have any questions or need further assistance, please don’t hesitate to reach out. 

Best regards,  
[Your Name]  
Customer Success Manager

---
## 7. Same Question, Different Models + Real Cost Tracking

In [7]:
total_cost = 0.0

def compare_models(prompt):
    global total_cost
    models = ["gpt-4o", "anthropic/claude-haiku-4-5"]
    
    for model in models:
        try:
            resp = completion(model=model, messages=[{"role": "user", "content": prompt}], temperature=0.0)
            
            prompt_tokens = getattr(resp.usage, 'prompt_tokens', 0)
            completion_tokens = getattr(resp.usage, 'completion_tokens', 0)
            cost = litellm.completion_cost(resp)
            total_cost += cost
            
            banner(f"Model: {model}", color=GOLD)
            print(f"Input tokens : {prompt_tokens}")
            print(f"Output tokens: {completion_tokens}")
            print(f"This call    : ${cost:.6f}")
            print(f"Running total: ${total_cost:.6f}")
            display(Markdown(resp.choices[0].message.content.strip()))
            print("─" * 80)
            
        except Exception as e:
            print(f"Error with {model}: {e}")

compare_models("Summarize the key benefits and risks of using generative AI in our company.")

Input tokens : 24
Output tokens: 475
This call    : $0.004810
Running total: $0.004810


Using generative AI in your company can offer several key benefits, but it also comes with certain risks. Here's a summary of both:

### Benefits:

1. **Increased Efficiency and Productivity:**
   - Automates repetitive tasks, freeing up employees to focus on more strategic activities.
   - Speeds up content creation, data analysis, and other processes, leading to faster decision-making.

2. **Enhanced Creativity and Innovation:**
   - Generates new ideas, designs, and solutions that might not be immediately apparent to human teams.
   - Facilitates brainstorming and prototyping by providing diverse options and perspectives.

3. **Personalization and Customer Engagement:**
   - Creates personalized content and recommendations, improving customer experience and satisfaction.
   - Enhances marketing strategies by tailoring messages to specific audience segments.

4. **Cost Savings:**
   - Reduces the need for extensive human resources in certain areas, lowering operational costs.
   - Minimizes errors and rework, saving time and resources.

5. **Scalability:**
   - Easily scales operations without a proportional increase in costs, allowing for rapid growth and adaptation.

### Risks:

1. **Quality and Accuracy Concerns:**
   - May produce outputs that are inaccurate or of low quality, requiring human oversight and validation.
   - Risk of generating biased or inappropriate content if the AI is trained on biased data.

2. **Security and Privacy Issues:**
   - Potential for data breaches or misuse of sensitive information if AI systems are not properly secured.
   - Compliance challenges with data protection regulations, such as GDPR.

3. **Dependence and Skill Gaps:**
   - Over-reliance on AI could lead to a loss of critical skills among employees.
   - Requires ongoing training and upskilling to effectively integrate AI into workflows.

4. **Ethical and Legal Challenges:**
   - Concerns about intellectual property rights, especially if AI-generated content is used commercially.
   - Ethical dilemmas related to the use of AI in decision-making processes.

5. **Implementation Costs:**
   - Initial investment in AI technology and infrastructure can be high.
   - Continuous maintenance and updates are necessary to keep AI systems effective and secure.

Balancing these benefits and risks involves careful planning, robust governance frameworks, and ongoing monitoring to ensure that generative AI is used responsibly and effectively within your company.

────────────────────────────────────────────────────────────────────────────────


Input tokens : 24
Output tokens: 318
This call    : $0.001614
Running total: $0.006424


# Generative AI: Key Benefits and Risks

## Benefits

**Efficiency & Productivity**
- Automate routine tasks (writing, coding, analysis, customer service)
- Faster content creation and iteration
- Reduced time on repetitive work

**Cost Reduction**
- Lower labor costs for certain functions
- Fewer resources needed for some processes

**Innovation & Quality**
- New product/service possibilities
- Enhanced decision-making with data analysis
- Improved customer experiences

**Scalability**
- Handle increased volume without proportional headcount growth

## Risks

**Quality & Accuracy**
- Hallucinations and factual errors
- Requires human review and verification
- Context misunderstandings

**Security & Privacy**
- Data exposure if sensitive information enters AI systems
- Intellectual property concerns
- Compliance issues (GDPR, industry regulations)

**Operational**
- Over-reliance on AI outputs
- Integration challenges with existing systems
- Skill gaps in your workforce

**Reputational & Legal**
- Bias in outputs affecting customers/decisions
- Liability for AI-generated errors
- Unclear accountability

**Workforce**
- Job displacement concerns
- Need for reskilling employees

## Recommendation
Start with **low-risk pilots** in controlled areas, establish clear governance policies, and invest in staff training before broad deployment.

What specific use cases are you considering?

────────────────────────────────────────────────────────────────────────────────


---
## Final Takeaways

1. **LLMs predict tokens**, they don’t retrieve facts.
2. **Tokens = money** — longer documents cost more.
3. Use **temperature** deliberately: low for precision, higher for creativity.
4. **Hallucinations are inevitable** — always verify critical content.
5. **System prompts** are your most powerful control tool.
6. Different models = different strengths and costs.
7. Combine **great prompting + human oversight** for best results.

**You don’t need to become a prompt engineer — you just need to know what to watch for.**